# Uncertainty Analysis: Brightness
This notebook analyzes the impact of image brightness variations on slope estimation accuracy.
It includes:
1.  **Data Preparation**: Adjusting brightness of panoramas (simulating exposure changes).
2.  **Image Transformation**: Converting adjusted panoramas to perspective views.
3.  **Result Analysis**: Comparing estimated slopes against ground truth.
4.  **Visualization**: Plotting uncertainty and data availability.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import concurrent.futures
from pathlib import Path
from tqdm import tqdm
from zensvi.transform import ImageTransformer

# Plotting settings
plt.rcParams.update({'font.size': 14})
plt.rcParams['font.family'] = 'Arial'

In [ ]:
# Configuration & Paths
BASE_DIR = 'uncertainty_analysis'

# Input Data
VALIDATION_SAMPLES_PATH = os.path.join(BASE_DIR, 'validation/inliers_05deg_sample.gpkg')
SOURCE_PANOS_DIR = os.path.join(BASE_DIR, 'degradation_test/panoramas')

# Working Directories
DEGRADATION_DIR = os.path.join(BASE_DIR, 'brightness')
BRIGHTNESS_PANOS_DIR = os.path.join(DEGRADATION_DIR, 'brightness-panos')
TRANSFORMED_DIR = os.path.join(DEGRADATION_DIR, 'brightness_panos-transformed')
RESULTS_DIR = os.path.join(DEGRADATION_DIR, 'brightness-panos_test')

# Output
FIGURE_SAVE_PATH = os.path.join(BASE_DIR, 'brightness_uncertainty.png')

# Parameters
# Brightness factors: < 1 for darker, > 1 for brighter
# Mapped to levels: -2, -1, 0 (original), 1, 2, 3
BRIGHTNESS_LEVELS = list(range(-2, 4)) 
# Note: The original code logic for mapping levels to factors is a bit complex, 
# involving different loops for positive and negative levels. 
# We will standardize this in the processing function.

In [ ]:
# Load validation samples
subsamples = gpd.read_file(VALIDATION_SAMPLES_PATH)
display(subsamples.head())

In [ ]:
def adjust_brightness(image_path, output_dir, factor):
    """
    Adjust the brightness of an image by a specified factor.
    """
    image = cv2.imread(str(image_path))
    if image is None:
        return f"Error: Could not read {image_path}"

    # Convert to float32 for precision
    image_float = image.astype(np.float32)

    # Adjust brightness
    adjusted_image = cv2.convertScaleAbs(image_float, alpha=factor)

    # Save
    output_path = Path(output_dir) / Path(image_path).name
    cv2.imwrite(str(output_path), adjusted_image)
    return None

def generate_brightness_panoramas(source_dir, output_base, levels):
    """
    Generate brightness-adjusted panoramas.
    Mapping logic based on original code:
    Level > 0: factor = level + 1 (e.g., Level 1 -> factor 2)
    Level < 0: factor = 1 / (abs(level) + 1) ? Or based on original code logic?
    
    Original code:
    - Positive loop: range(2, 5, 1) -> factors 2, 3, 4. Output dirs: 2, 3, 4.
    - Negative loop logic was separate.
    
    Let's standardize:
    Level 0: Factor 1.0 (Original)
    Level 1: Factor 2.0
    Level 2: Factor 3.0
    Level 3: Factor 4.0
    Level -1: Factor 0.5 (1/2)
    Level -2: Factor 0.33 (1/3)
    """
    png_files = list(Path(source_dir).glob("*.png"))
    print(f"Found {len(png_files)} source images.")
    
    from concurrent.futures import ProcessPoolExecutor
    
    futures = []
    with ProcessPoolExecutor() as executor:
        for level in levels:
            if level == 0:
                continue # Skip original
                
            if level > 0:
                factor = level + 1.0
            else:
                factor = 1.0 / (abs(level) + 1.0)
                
            output_dir = os.path.join(output_base, str(level))
            os.makedirs(output_dir, exist_ok=True)
            
            print(f"Generating Level {level} (Factor {factor:.2f}) -> {output_dir}")
            
            for img_path in png_files:
                futures.append(executor.submit(adjust_brightness, img_path, output_dir, factor))
                
        # Wait for completion
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
            res = future.result()
            if res: print(res)

generate_brightness_panoramas(SOURCE_PANOS_DIR, BRIGHTNESS_PANOS_DIR, BRIGHTNESS_LEVELS)

In [ ]:
def run_brightness_transformation(input_base, output_base, levels):
    """Transform brightness-adjusted panoramas to perspective images."""
    for level in levels:
        if level == 0: continue
        
        input_dir = os.path.join(input_base, str(level))
        # Note: Original code used specific naming like 'chunk_{i}' or '-{i}'
        # We will use a consistent 'chunk_{level}' format for output
        output_dir = os.path.join(output_base, f'chunk_{level}')
        
        if not os.path.exists(input_dir):
            print(f"Skipping Level {level}: Input directory not found.")
            continue
            
        print(f"Transforming Level {level} -> {output_dir}")
        image_transformer = ImageTransformer(dir_input=input_dir, dir_output=output_dir)
        image_transformer.transform_images(
            style_list="perspective",
            FOV=90,
            theta=90,
            phi=0,
            aspects=(10, 10),
            show_size=100
        )
        
        # Cleanup
        os.system(f"find {output_dir} -name '*Direction_0*' -delete")
        os.system(f"find {output_dir} -name '*Direction_180*' -delete")

run_brightness_transformation(BRIGHTNESS_PANOS_DIR, TRANSFORMED_DIR, BRIGHTNESS_LEVELS)

In [ ]:
def search_csv_files(directory, keyword=None):
    csv_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                if keyword is None or keyword in file:
                    csv_files.append(os.path.join(root, file))
    return csv_files

def load_brightness_results(base_path, levels, sample_df):
    """Load and merge results from different brightness runs."""
    merged_df = sample_df[['pano_id', 'slope_1m', 'slope_10m', 'slope_30m']].copy()
    merged_df = merged_df.set_index('pano_id')
    
    available_ratios = []
    
    for level in levels:
        # Handle Level 0 (Original) - assuming it's the baseline or handled separately?
        # The original code calculates diff against slope_1m, so we just need the adjusted results for other levels.
        if level == 0:
            # For level 0, we might want to use the original results if available, 
            # or just skip if we are only looking at diffs from ground truth.
            # Let's assume we skip loading level 0 result file specifically for now 
            # and just use slope_1m as reference, but we need availability.
            # If you have a specific result file for Level 0, add logic here.
            available_ratios.append(100.0) # Assuming original is 100%
            continue

        search_path = os.path.join(base_path, f'chunk_{level}')
        csv_files = search_csv_files(search_path, keyword='adjusted')
        
        if csv_files:
            res_df = pd.read_csv(csv_files[0])
            
            n = len(res_df)
            available_ratios.append(n / 2000 * 100) 
            
            res_df.set_index('pano_id', inplace=True)
            res_df = res_df[['adjusted_angle']]
            res_df = res_df.rename(columns={'adjusted_angle': f'brightness_{level}_adjusted_angle'})
            
            res_df = res_df[~res_df.index.duplicated(keep='first')]
            
            merged_df = merged_df.join(res_df, how='left')
            
            merged_df[f'diff_brightness_1m_{level}'] = merged_df['slope_1m'] - merged_df[f'brightness_{level}_adjusted_angle']
        else:
            print(f"No results found for Level {level}")
            available_ratios.append(0)
            
    return merged_df, available_ratios

analyzed_df, availability = load_brightness_results(RESULTS_DIR, BRIGHTNESS_LEVELS, subsamples)
display(analyzed_df.head())

In [ ]:
def plot_brightness_summary(sample_df, availability_pct, levels, save_path=None):
    import matplotlib.ticker as mticker
    
    # Filter valid levels
    plot_levels = [l for l in levels if f'diff_brightness_1m_{l}' in sample_df.columns]
    if not plot_levels:
        print("No data to plot.")
        return

    rename_map = {f'diff_brightness_1m_{l}': f"{l}" for l in plot_levels}
    plot_df = (
        sample_df[[f'diff_brightness_1m_{l}' for l in plot_levels]]
        .rename(columns=rename_map)
        .melt(var_name='Brightness Level', value_name='Slope difference (°)')
    )
    
    # X-axis labels
    x_labels_map = {
        -2: 'Underexposed', 
        -1: 'Slightly\nunderexposed', 
        0: 'Normal', 
        1: 'Slightly\noverexposed', 
        2: 'Overexposed',
        3: 'Highly\noverexposed'
    }
    order_labels = [str(l) for l in plot_levels]
    display_labels = [x_labels_map.get(int(l), str(l)) for l in order_labels]

    # Plotting
    sns.set_theme(style='whitegrid', context='talk')
    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[2, 1], hspace=0.32)
    
    # Boxplot
    ax_box = fig.add_subplot(gs[0])
    sns.boxplot(
        data=plot_df,
        x='Brightness Level',
        y='Slope difference (°)',
        ax=ax_box,
        width=0.35,
        fliersize=2.2,
        linewidth=1.2,
        medianprops={'color': '#0F3460', 'linewidth': 2},
        whiskerprops={'color': '#0F3460', 'linewidth': 1.2},
        capprops={'color': '#0F3460', 'linewidth': 1.2},
        showfliers=False,
        showcaps=False,
        palette=sns.cubehelix_palette(len(order_labels), start=0.35, rot=-0.15, light=0.85, dark=0.25)
    )
    
    ax_box.axhline(0, color='#5B6270', linestyle='--', linewidth=1, alpha=0.8, zorder=0)
    ax_box.set_xlabel('')
    ax_box.set_ylabel('Slope difference (°)', labelpad=12)
    ax_box.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}'))
    ax_box.set_xticklabels(display_labels)
    
    avail_dict = dict(zip(levels, availability_pct))
    plot_avail = [avail_dict[l] for l in plot_levels]
    
    positions = np.arange(len(order_labels))
    
    ax_line = fig.add_subplot(gs[1], sharex=ax_box)
    ax_line.plot(positions, plot_avail, color=sns.color_palette('crest', n_colors=7)[-2], marker='o', linewidth=2.4, label='Availability')
    ax_line.fill_between(positions, plot_avail, color=sns.color_palette('crest', n_colors=7)[-2], alpha=0.15)
    
    ax_line.set_xlabel('Exposure level', labelpad=12)
    ax_line.set_ylabel('Available imagery (%)', labelpad=12)
    ax_line.set_xticks(positions)
    ax_line.set_xticklabels(display_labels)
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Plot
plot_brightness_summary(
    analyzed_df,
    availability,
    BRIGHTNESS_LEVELS,
    save_path=FIGURE_SAVE_PATH
)